In [ ]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase


# -----------------------------------------------------
# 1. 환경변수 로드
# -----------------------------------------------------

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD가 .env에 설정되어 있지 않습니다.")

ValueError: NEO4J_PASSWORD가 .env에 설정되어 있지 않습니다.

In [ ]:



# -----------------------------------------------------
# 2. Neo4j 연결
# -----------------------------------------------------

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)


# -----------------------------------------------------
# 3. 제약조건 생성
# -----------------------------------------------------

CONSTRAINT_QUERIES = [
    """
    CREATE CONSTRAINT recipe_uid_unique IF NOT EXISTS
    FOR (r:Recipe)
    REQUIRE r.recipe_uid IS UNIQUE
    """,

    """
    CREATE CONSTRAINT dish_name_unique IF NOT EXISTS
    FOR (d:Dish)
    REQUIRE d.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT component_uid_unique IF NOT EXISTS
    FOR (c:Component)
    REQUIRE c.component_uid IS UNIQUE
    """,

    """
    CREATE CONSTRAINT ingredient_name_normalized_unique IF NOT EXISTS
    FOR (i:Ingredient)
    REQUIRE i.name_normalized IS UNIQUE
    """,
]


def create_constraints():
    with driver.session() as session:
        for query in CONSTRAINT_QUERIES:
            session.run(query)

        result = session.run(
            """
            SHOW CONSTRAINTS
            YIELD name, type, entityType, labelsOrTypes, properties
            RETURN name, type, entityType, labelsOrTypes, properties
            ORDER BY name
            """
        )

        return [record.data() for record in result]


# -----------------------------------------------------
# 4. 실행
# -----------------------------------------------------

if __name__ == "__main__":
    try:
        driver.verify_connectivity()
        print(f"Neo4j 연결 성공: {NEO4J_URI}")

        constraints = create_constraints()

        print("\n현재 생성된 제약조건:")
        for constraint in constraints:
            print(constraint)

    finally:
        driver.close()